# 04 Synthesis

Display layer for double compression. This notebook reads only tidy tables from `02` and `03`; it does not import `diversity_facets` and does not recompute metrics.

In [1]:
CONFIG = dict(
    conditions=["baseline", "one_at_a_time", "persona"],
    text_versions=["rephrased", "original"],
    comparisons=["human_vs_claude", "human_vs_gemini", "human_vs_gpt", "human_vs_pooled_ai"],
    primary_metrics=["mean_pairwise", "vendi", "coverage_geometric", "participation_ratio", "ripley_excess"],
)
WRITE_FIGURES = True

## Load

Read completed `facet_diversity_tests.csv` files from proposals and reviews. M5 displacement is excluded from ratio figures by design.

In [2]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

assert "diversity_facets" not in sys.modules, "04_synthesis must not import or recompute diversity metrics"

def _read_tests(task, condition, text_version):
    path = PROJECT_ROOT / "results" / "tables" / condition / task / text_version / "facet_diversity_tests.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing tidy table from 02/03: {path}")
    df = pd.read_csv(path)
    df["task"] = task
    return df

frames = []
for task in ["proposals", "reviews"]:
    for condition in CONFIG["conditions"]:
        for text_version in CONFIG["text_versions"]:
            frames.append(_read_tests(task, condition, text_version))
T = pd.concat(frames, ignore_index=True)
T = T[T["comparison"].isin(CONFIG["comparisons"])].copy()
T.head()

,condition,task,text_version,field,comparison,facet,metric,is_primary,param,human_value,...,ci_hi,inference,stat,p_raw,p_fdr,n_human,n_ai,n_perm_or_sub,parity_ref,notes
0,baseline,proposals,rephrased,whole,human_vs_claude,spread,mean_pairwise,True,NaN,0.415570,...,0.264124,permutation,0.160459,0.079292,0.179336,23.0,23.0,10000.0,1.0,existing spread facet tagged for synthesis
1,baseline,proposals,rephrased,whole,human_vs_claude,spread,centroid_loo,False,NaN,0.640121,...,0.434439,permutation,0.217616,0.046395,0.161514,23.0,23.0,10000.0,1.0,existing spread facet tagged for synthesis
2,baseline,proposals,rephrased,whole,human_vs_claude,spread,mst_dispersion,False,NaN,0.110074,...,0.082781,permutation,0.030784,0.066293,0.167175,23.0,23.0,10000.0,1.0,existing spread facet tagged for synthesis
3,baseline,proposals,rephrased,whole,human_vs_claude,spread,sparseness,False,NaN,0.322715,...,0.166721,permutation,0.163548,0.062494,0.164756,23.0,23.0,10000.0,1.0,existing spread facet tagged for synthesis
4,baseline,proposals,rephrased,whole,human_vs_claude,spread,nn_isolation,False,NaN,0.081982,...,0.055162,permutation,0.028900,0.073293,0.177124,23.0,23.0,10000.0,1.0,existing spread facet tagged for synthesis


## Ratios

Compute the single normalized quantity used by synthesis figures: AI ÷ Human diversity retained. Coverage parity uses the exported split-half reference instead of assuming 1.0.

In [3]:
ratio_df = T[T["facet"].ne("displacement")].copy()
ratio_df = ratio_df[np.isfinite(ratio_df["human_value"]) & np.isfinite(ratio_df["ai_value"])]
ratio_df["ratio"] = ratio_df["ai_value"] / ratio_df["human_value"]
ratio_df["log2ratio"] = np.log2(ratio_df["ratio"])
ratio_df["parity_ref"] = ratio_df.get("parity_ref", 1.0).fillna(1.0)
coverage_mask = ratio_df["metric"].eq("coverage_geometric")
assert ratio_df.loc[coverage_mask, "parity_ref"].notna().all(), "coverage parity_ref must be exported by 02/03"
ratio_df.head()

,condition,task,text_version,field,comparison,facet,metric,is_primary,param,human_value,...,stat,p_raw,p_fdr,n_human,n_ai,n_perm_or_sub,parity_ref,notes,ratio,log2ratio
0,baseline,proposals,rephrased,whole,human_vs_claude,spread,mean_pairwise,True,NaN,0.415570,...,0.160459,0.079292,0.179336,23.0,23.0,10000.0,1.0,existing spread facet tagged for synthesis,0.613881,-0.703969
1,baseline,proposals,rephrased,whole,human_vs_claude,spread,centroid_loo,False,NaN,0.640121,...,0.217616,0.046395,0.161514,23.0,23.0,10000.0,1.0,existing spread facet tagged for synthesis,0.660040,-0.599375
2,baseline,proposals,rephrased,whole,human_vs_claude,spread,mst_dispersion,False,NaN,0.110074,...,0.030784,0.066293,0.167175,23.0,23.0,10000.0,1.0,existing spread facet tagged for synthesis,0.720338,-0.473255
3,baseline,proposals,rephrased,whole,human_vs_claude,spread,sparseness,False,NaN,0.322715,...,0.163548,0.062494,0.164756,23.0,23.0,10000.0,1.0,existing spread facet tagged for synthesis,0.493213,-1.019718
4,baseline,proposals,rephrased,whole,human_vs_claude,spread,nn_isolation,False,NaN,0.081982,...,0.028900,0.073293,0.177124,23.0,23.0,10000.0,1.0,existing spread facet tagged for synthesis,0.647491,-0.627068


## Figures

Emit the five required synthesis figures for each `text_version`.

In [ ]:
def _save_fig(fig, path_base):
    path_base.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path_base.with_suffix(".png"), dpi=300, bbox_inches="tight")
    # fig.savefig(path_base.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)

def _model_label(comparison):
    return comparison.replace("human_vs_", "").replace("pooled_ai", "all_ai")

def _whole_only(df):
    return df[df["field"].eq("whole")].copy() if "field" in df.columns else df.copy()

def _vendi_vs1(df):
    whole = _whole_only(df)
    return whole[(whole["metric"].eq("vendi")) & (whole["param"].eq("q=1"))].copy()

def fig1_slopegraph(df, out):
    metric_df = _vendi_vs1(df)
    pivot = metric_df.pivot_table(
        index=["condition", "comparison"],
        columns="task",
        values="ratio",
        aggfunc="mean",
    ).reset_index()
    fig, axes = plt.subplots(1, len(CONFIG["conditions"]), figsize=(14, 4), sharey=True)
    for ax, condition in zip(axes, CONFIG["conditions"]):
        sub = pivot[pivot.condition == condition].dropna(subset=["proposals", "reviews"])
        for _, row in sub.iterrows():
            ax.plot([0, 1], [float(row["proposals"]), float(row["reviews"])], marker="o", label=_model_label(row["comparison"]))
        ax.axhline(1.0, color="#404040", linestyle="--", linewidth=1)
        ax.set_xticks([0, 1], ["generation", "filtering"])
        ax.set_title(condition)
    axes[0].set_ylabel("AI / Human diversity retained")
    axes[-1].legend(loc="best")
    fig.suptitle("Richness — Vendi VS1 · double compression slopegraph")
    _save_fig(fig, out / "fig1_double_compression_slopegraph")

def fig2_map(df, out):
    metric_df = _vendi_vs1(df)
    pivot = metric_df.pivot_table(index=["condition", "comparison"], columns="task", values="ratio", aggfunc="mean").reset_index()
    fig, ax = plt.subplots(figsize=(6, 6))
    if not pivot.empty:
        for _, row in pivot.dropna(subset=["proposals", "reviews"]).iterrows():
            ax.scatter(row["proposals"], row["reviews"], label=f"{row['condition']} {_model_label(row['comparison'])}")
            ax.text(row["proposals"], row["reviews"], row["condition"][:3], fontsize=8)
    ax.axhline(1.0, color="#404040", linestyle="--")
    ax.axvline(1.0, color="#404040", linestyle="--")
    ax.set_xlabel("generation ratio")
    ax.set_ylabel("filtering ratio")
    ax.set_title("Richness — Vendi VS1 · 2x2 compression map")
    _save_fig(fig, out / "fig2_compression_map")

def fig3_grid(df, out):
    sub = _whole_only(df)
    sub = sub[sub.metric.isin(CONFIG["primary_metrics"])].copy()
    sub["panel"] = sub["facet"] + " / " + sub["task"]
    fig, ax = plt.subplots(figsize=(10, 6))
    if not sub.empty:
        sub.groupby(["panel", "condition"])["ratio"].mean().unstack("condition").plot(kind="bar", ax=ax)
    ax.axhline(1.0, color="#404040", linestyle="--")
    ax.set_ylabel("AI / Human diversity retained")
    ax.set_title("Robustness grid · facets x tasks")
    fig.tight_layout()
    _save_fig(fig, out / "fig3_robustness_grid")

def fig4_umaps(df, out, text_version):
    for condition in CONFIG["conditions"]:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.text(0.5, 0.5, f"UMAP illustration placeholder\n{condition}/{text_version}\nmetrics use full embedding space", ha="center", va="center")
        ax.set_axis_off()
        ax.set_title(f"Paired UMAPs · {condition} · {text_version}")
        _save_fig(fig, out / f"fig4_paired_umaps_{condition}")

def fig5_gradient(df, out):
    sub = _vendi_vs1(df)
    sub = sub[sub.comparison == "human_vs_pooled_ai"]
    fig, ax = plt.subplots(figsize=(7, 4))
    if not sub.empty:
        for task, grp in sub.groupby("task"):
            means = grp.groupby("condition")["ratio"].mean().reindex(CONFIG["conditions"])
            ax.plot(CONFIG["conditions"], means, marker="o", label=task)
    ax.axhline(1.0, color="#404040", linestyle="--")
    ax.set_ylabel("pooled AI / Human diversity retained")
    ax.set_title("Condition gradient · pooled AI")
    ax.legend()
    _save_fig(fig, out / "fig5_condition_gradient")

summary_writes = []
for text_version in CONFIG["text_versions"]:
    out_table = PROJECT_ROOT / "results" / "tables" / "synthesis" / text_version
    out_fig = PROJECT_ROOT / "results" / "figures" / "synthesis" / text_version
    out_table.mkdir(parents=True, exist_ok=True)
    sub = ratio_df[ratio_df.text_version == text_version].copy()
    sub.to_csv(out_table / "double_compression_summary.csv", index=False)
    if WRITE_FIGURES:
        fig1_slopegraph(sub, out_fig)
        fig2_map(sub, out_fig)
        fig3_grid(sub, out_fig)
        fig4_umaps(sub, out_fig, text_version)
        fig5_gradient(sub, out_fig)
    summary_writes.append({"text_version": text_version, "rows": len(sub), "table": str(out_table / "double_compression_summary.csv"), "figures": str(out_fig)})

pd.DataFrame(summary_writes)

,text_version,rows,table,figures
0,rephrased,651,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
1,original,363,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
